[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C06_Interpretability_Course/02_logit_lens/02_logit_lens.ipynb)

# 02 · Residual Stream 解剖与 Logit Lens

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 numpy + matplotlib，所有 cell 秒级运行，无需任何深度学习框架。

**本 notebook 你将完成：**

1. **手工构造**（不训练！）一个 2 层 attention-only 的 numpy 迷你 transformer：previous-token head + induction head，让它在"重复序列 copy"任务上正确预测下一个 token；
2. 实现 **logit lens**：把每层 residual stream 直接过 final RMSNorm + unembedding，画 layer × position 热图，看预测如何逐层"成形"；
3. 实现 **direct logit attribution (DLA)**：把最终 logit 精确分解为 embedding / head 0 / head 1 各自的写入贡献；
4. 对比**带与不带 final norm** 的 lens，理解 RMSNorm/LayerNorm 对分析结论的影响；
5. 完成 **4 道 ✏️ 练习**（causal attention 权重、`logit_lens`、`direct_logit_attribution`、KL 收敛度量），每道附 assert 自动判分。

参考文献：Elhage et al. 2021 (*A Mathematical Framework for Transformer Circuits*) · nostalgebraist 2020 (*interpreting GPT: the logit lens*) · Belrose et al. 2023 (*tuned lens*, arXiv:2303.08112) · Lindsey et al. 2025 (*On the Biology of a Large Language Model*)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def rms_norm(x, gamma=None, eps=1e-6):
    """RMSNorm: x / rms(x) * gamma。gamma=None 视为全 1（恒等缩放）。"""
    rms = np.sqrt((x ** 2).mean(axis=-1, keepdims=True) + eps)
    out = x / rms
    if gamma is not None:
        out = out * gamma
    return out

print("numpy", np.__version__)

## 1 · 手工构造一个 2 层 attention-only transformer

**任务：重复序列 copy。** 输入形如 `H A B C D E F | A B C D E F | A`——一段序列重复出现。
第二遍读到 `A` 时，模型应该预测 `B`（上次 `A` 后面跟的就是 `B`）。这是 [Elhage 2021] 中
**induction head** 的最小工作场景，我们不训练，直接**手写权重**把电路搭出来。

把 $d_{model}=32$ 维 residual stream 划成三个互不重叠的子空间（这是手工电路的奢侈——真实模型里子空间是叠加、纠缠的）：

| dims | 子空间 | 谁写入 | 谁读取 |
|------|--------|--------|--------|
| 0–7   | 当前 token one-hot（vocab=8） | $W_E$；head 1 的 $W_O$ 也写回这里 | head 1 的 Q/V；**unembedding $W_U$ 只读这里** |
| 8–23  | position one-hot（16 个位置） | $W_P$ | head 0 的 Q/K |
| 24–31 | PREV："我前面那个 token 是什么" | head 0 的 $W_O$ | head 1 的 K |

**两层电路的分工**（每个 head 都是"从 stream 读 → 算 → 加回 stream"的加性组件）：

```
layer 0  previous-token head:
  Q: 位置 p 的 query = "位置 p-1" 的 one-hot     K: 位置 one-hot
  → 注意力打到前一个位置;  V/O 把那里的 token 身份搬进 PREV 子空间(dims 24-31)
layer 1  induction head:
  Q: 当前 token one-hot                          K: PREV 子空间
  → 注意到 "上一次当前 token 出现之后的那个位置" j
  V/O: 把位置 j 的 token 身份 × ALPHA(=4) 写回 token 子空间(dims 0-7) → 主导 logits
```

注意 direct path：$W_E$ 把当前 token 写在 dims 0–7，而 $W_U$ 恰好读 dims 0–7——
所以"什么都不做"时模型预测**当前 token 自己**（一个退化的 bigram）。induction head 用 4 倍幅度的写入压过它。

In [ ]:
V, P, D, DH = 8, 16, 32, 16          # vocab / 最大位置数 / d_model / d_head
TOKS = list("ABCDEFGH")
S0, S1, ALPHA = 12.0, 12.0, 4.0      # 两个 head 的 QK 打分增益; induction 写回增益

# 嵌入与读出：token one-hot 占 dims 0-7, position one-hot 占 dims 8-23
W_E = np.zeros((V, D)); W_E[np.arange(V), np.arange(V)] = 1.0
W_P = np.zeros((P, D)); W_P[np.arange(P), 8 + np.arange(P)] = 1.0
W_U = np.zeros((D, V)); W_U[np.arange(V), np.arange(V)] = 1.0   # unembedding 只读 token 子空间

# ---- layer 0: previous-token head ----
W_Q0 = np.zeros((D, DH)); W_K0 = np.zeros((D, DH))
for p in range(1, P):
    W_Q0[8 + p, p - 1] = S0          # 位置 p 的 query = "位置 p-1" 的 one-hot（放大 S0 倍）
for p in range(P):
    W_K0[8 + p, p] = 1.0             # 位置 p 的 key   = "位置 p"   的 one-hot
W_V0 = np.zeros((D, DH)); W_V0[np.arange(V), np.arange(V)] = 1.0       # 取被注意位置的 token 身份
W_O0 = np.zeros((DH, D)); W_O0[np.arange(V), 24 + np.arange(V)] = 1.0  # 写入 PREV 子空间 (24-31)

# ---- layer 1: induction head ----
W_Q1 = np.zeros((D, DH)); W_Q1[np.arange(V), np.arange(V)] = S1        # 读当前 token (0-7)
W_K1 = np.zeros((D, DH)); W_K1[24 + np.arange(V), np.arange(V)] = 1.0  # 读 PREV 子空间 (24-31)
W_V1 = np.zeros((D, DH)); W_V1[np.arange(V), np.arange(V)] = 1.0       # 取被注意位置的 token 身份
W_O1 = np.zeros((DH, D)); W_O1[np.arange(V), np.arange(V)] = ALPHA     # 放大 4 倍写回 token 子空间

LAYERS = [(W_Q0, W_K0, W_V0, W_O0), (W_Q1, W_K1, W_V1, W_O1)]
print("所有权重手工构造完成（没有任何训练）")

In [ ]:
def attention(xn, W_Q, W_K, W_V, W_O):
    """单 head 因果注意力。xn: 已归一化的 residual (n, D)。返回 (写回向量, 注意力矩阵)。"""
    q, k, v = xn @ W_Q, xn @ W_K, xn @ W_V
    scores = q @ k.T / np.sqrt(DH)
    n = xn.shape[0]
    mask = np.triu(np.ones((n, n), dtype=bool), k=1)   # j > i 的未来位置
    scores = np.where(mask, -1e9, scores)
    A = softmax(scores, axis=-1)
    return A @ v @ W_O, A

def forward(tokens):
    """pre-norm 前向；cache 记录每阶段 residual、每个组件的写入、注意力。"""
    n = len(tokens)
    x = W_E[tokens] + W_P[np.arange(n)]
    cache = {"resid": [x.copy()], "writes": [x.copy()], "attn": []}   # writes[0] = embedding(直接路径)
    for (W_Q, W_K, W_V, W_O) in LAYERS:
        out, A = attention(rms_norm(x), W_Q, W_K, W_V, W_O)
        x = x + out                          # 加性写回：stream 从不被整体重写
        cache["resid"].append(x.copy())
        cache["writes"].append(out)
        cache["attn"].append(A)
    logits = rms_norm(x) @ W_U               # final RMSNorm + unembedding
    return logits, cache

# 重复序列：H | A B C D E F | A B C D E F | A   （第二遍起每一步都可由 induction 预测）
tokens = np.array([7, 0, 1, 2, 3, 4, 5, 0, 1, 2, 3, 4, 5, 0])
logits, cache = forward(tokens)
pred = logits.argmax(-1)

print("输入 :", " ".join(TOKS[t] for t in tokens))
print("预测 :", " ".join(TOKS[t] for t in pred))
print("layer0 注意力 A[i, i-1]（应全为 1，previous-token head）:",
      np.round(np.diagonal(cache["attn"][0], -1), 4))

assert (pred[7:13] == tokens[8:14]).all(), "第二遍序列应被正确 copy"
assert pred[13] == 1, "末位 A 应预测 B"
print("\n✓ 手工 induction 电路工作正常：第二遍出现的 token 全部预测正确")

## 2 · Logit Lens：模型中途在想什么

[nostalgebraist 2020] 的想法：最终预测是 $\mathrm{LN}_f(x^{(L)})W_U$，那就把**每一层**的 residual stream
都过同一个 final norm + unembedding：

$$\mathrm{LogitLens}^{(\ell)} = \mathrm{LN}_f\big(x^{(\ell)}\big)\, W_U$$

得到"如果模型在第 $\ell$ 层就交卷会答什么"。我们的模型有 3 个阶段可看：embedding 后、layer 0 后、layer 1 后。

**预读热图前先押注**：
- 阶段 0（embedding）：$W_U$ 读 dims 0–7，那里只有当前 token → lens 预测**当前 token 自己**；
- 阶段 1（+layer 0）：previous-token head 把全部工作写进 dims 24–31——与 $W_U$ 读取空间**正交**，
  lens 应该**完全看不到变化**（这是 logit lens 的经典盲区，也是 tuned lens 要修的问题）；
- 阶段 2（+layer 1）：induction head 以 4 倍幅度写回 token 子空间 → 第二遍序列的预测瞬间"成形"。

In [ ]:
resid_stack = np.stack(cache["resid"])            # (3 阶段, n, D)
STAGES = ["embed", "+layer0", "+layer1"]

lens = rms_norm(resid_stack) @ W_U                # logit lens: (3, n, V)
assert np.allclose(lens[-1], logits)              # 自检：末阶段必须 = 模型真实输出

# 每个位置的"正确答案"：下一个 token（末位 A 的 induction 答案是 B）
targets = np.append(tokens[1:], 1)
tgt_logit = np.take_along_axis(lens, targets[None, :, None], axis=2)[:, :, 0]

fig, ax = plt.subplots(figsize=(11, 3.2))
im = ax.imshow(tgt_logit, cmap="viridis", aspect="auto")
for s in range(len(STAGES)):                      # 每格标注该阶段 lens 的 top-1 预测
    top1 = lens[s].argmax(-1)
    for i in range(len(tokens)):
        ok = top1[i] == targets[i]
        ax.text(i, s, TOKS[top1[i]] + ("*" if ok else ""), ha="center", va="center",
                color="w", fontweight="bold" if ok else "normal", fontsize=9)
ax.set_xticks(range(len(tokens)), [f"{i}\n{TOKS[t]}" for i, t in enumerate(tokens)])
ax.set_yticks(range(len(STAGES)), STAGES)
ax.set_xlabel("position (输入 token)"); ax.set_title("logit lens: 各阶段对『正确下一 token』的 logit（格内 = top-1 预测，* = 命中）")
plt.colorbar(im, label="target logit"); plt.tight_layout(); plt.show()

print("阶段 0 top-1 =", " ".join(TOKS[t] for t in lens[0].argmax(-1)), " ← 全是当前 token（direct path）")
print("阶段 1 top-1 =", " ".join(TOKS[t] for t in lens[1].argmax(-1)), " ← 与阶段 0 相同：layer0 的工作 lens 看不见！")
print("阶段 2 top-1 =", " ".join(TOKS[t] for t in lens[2].argmax(-1)), " ← 第二遍序列 induction 预测成形")

## 3 · Direct Logit Attribution：这 8 分是谁写进去的

residual stream 是严格加性的：$x^{(L)} = e + o_0 + o_1$（embedding + head 0 写入 + head 1 写入）。
final RMSNorm 的缩放因子 $1/\mathrm{rms}(x^{(L)})$ 虽是非线性的，但**冻结**为真实最终 residual 上算出的值后，
logits 就精确分解为各组件贡献之和（gamma=1、无均值减法时这是恒等式，不是近似）：

$$\mathrm{logits} = \sum_{c \in \{e,\, o_0,\, o_1\}} \frac{c}{\mathrm{rms}(x^{(L)})}\, W_U$$

押注：对第二遍序列的正确 token，**embedding 贡献 0**（它推的是当前 token 不是下一个）、
**head 0 贡献恰好为 0**（写入子空间与 $W_U$ 正交——DLA 为 0 但它是 head 1 的供货商，删了它电路立刻瘫痪：
"直接贡献小 ≠ 不重要"，因果重要性要等模块 04 的 activation patching）、**head 1 贡献全部**。

In [ ]:
final_resid = cache["resid"][-1]
frozen_scale = 1.0 / np.sqrt((final_resid ** 2).mean(-1, keepdims=True) + 1e-6)  # 冻结的 1/rms

components = np.stack(cache["writes"])                 # (3, n, D): [embedding, head0, head1]
COMP = ["embedding (direct path)", "head 0 (prev-token)", "head 1 (induction)"]
contrib = (components * frozen_scale) @ W_U            # (3, n, V) 各组件对每个 logit 的直接贡献

assert np.allclose(contrib.sum(0), logits, atol=1e-9)  # 恒等式：各组件贡献之和 = 最终 logits
print("✓ DLA 分解恒等式成立: max|Σ贡献 - logits| =", np.abs(contrib.sum(0) - logits).max())

i = 7                                                  # 位置 7：第二遍的 A，正确答案 B
tgt = targets[i]
vals = contrib[:, i, tgt]
cur = contrib[:, i, tokens[i]]
fig, ax = plt.subplots(figsize=(7, 3))
xpos = np.arange(3)
ax.bar(xpos - 0.18, vals, 0.36, label=f"对正确答案 {TOKS[tgt]} 的 logit 贡献")
ax.bar(xpos + 0.18, cur, 0.36, label=f"对当前 token {TOKS[tokens[i]]} 的 logit 贡献")
ax.set_xticks(xpos, COMP, fontsize=8); ax.axhline(0, color="k", lw=0.5)
ax.set_title(f"DLA @ position {i}（输入 {TOKS[tokens[i]]}，应预测 {TOKS[tgt]}）")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

print(f"对正确答案 {TOKS[tgt]} 的贡献: embedding={vals[0]:.3f}  head0={vals[1]:.3f}  head1={vals[2]:.3f}")
print("head 0 对全词表所有位置的直接贡献最大绝对值:", np.abs(contrib[1]).max(), " ← 严格为 0（正交写入）")

## 4 · Final Norm 对 lens 的影响：什么变了，什么没变

RMSNorm（gamma 全 1）只是对**每个位置乘一个正标量** $1/\mathrm{rms}$——所以该位置 logits 的
**argmax 与排序完全不变**；变的是 logit 的绝对尺度，等价于改变 softmax 的**有效温度**。
residual 范数逐层增长（下面会打印），不带 norm 的 raw lens 会让深层 logit 虚高——
凡是涉及**概率 / 熵 / KL / 校准**的跨层结论，norm 的处理方式必须显式写明。

更危险的是真实模型：final norm 的 **gamma 不是均匀的**（LayerNorm 还要减均值），
此时跳过 norm 连 top-1 都可能改变。下面用一个非均匀 gamma 直接演示。

In [ ]:
raw_lens = resid_stack @ W_U                       # 不带 final norm 的 lens
rms_per_stage = np.sqrt((resid_stack ** 2).mean(-1))

print("各阶段 residual 的平均 rms:", np.round(rms_per_stage.mean(-1), 3), " ← 范数逐层增长")
print("raw lens 与 normed lens 的 top-1 是否处处一致:",
      (raw_lens.argmax(-1) == lens.argmax(-1)).all(), " ← gamma 均匀时排序不变\n")

# 但概率层面完全不同：看位置 7 上 top-1 token 的 softmax 概率
p_raw  = softmax(raw_lens[:, 7, :], axis=-1).max(-1)
p_norm = softmax(lens[:, 7, :],     axis=-1).max(-1)
for s in range(3):
    print(f"  {STAGES[s]:8s}  top-1 概率: raw={p_raw[s]:.3f}   normed={p_norm[s]:.3f}")
print("→ raw lens 的有效温度随 rms 漂移：浅层(rms<1)欠自信、深层(rms>1)过自信，跨层比较概率/熵时系统性失真\n")

# 非均匀 gamma（模拟真实模型学到的 final norm）：连 top-1 都会变
gamma_real = rng.uniform(0.3, 2.5, D)
lens_gamma = rms_norm(resid_stack, gamma_real) @ W_U
n_flip = (lens_gamma.argmax(-1) != lens.argmax(-1)).sum()
print(f"换成非均匀 gamma 后, {n_flip} 个 (阶段, 位置) 格子的 top-1 发生翻转",
      "← 真实模型里 final norm 不可跳过")

---
## ✏️ 练习 1：实现 `causal_attn_weights`

不翻上文，自己实现因果注意力权重：输入 `q, k`（形状均为 `(n, d_head)`），返回 `(n, n)` 注意力矩阵：
`scores = q @ k.T / sqrt(d_head)`，把未来位置（列号 > 行号）mask 掉，再按行 softmax。

**提示**：`np.triu(np.ones((n, n), dtype=bool), k=1)` 给出严格上三角 mask，对应位置填 `-1e9`（不要填 0——0 会参与 softmax！）；
`softmax` 直接用最上方定义好的。约 6 行。边界：第 0 行只能看自己，softmax 后应恰为 `[1, 0, ..., 0]`。

In [ ]:
def causal_attn_weights(q, k):
    # TODO: scores = q @ k.T / sqrt(d_head)
    # TODO: 用 -1e9 mask 掉严格上三角（未来位置）
    # TODO: 按行 softmax 后返回 (n, n) 矩阵
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
n = 5
k_t = np.eye(n) * 10.0                                  # key_j = 位置 j 的 one-hot
q_t = np.zeros((n, n)); q_t[np.arange(1, n), np.arange(n - 1)] = 10.0  # query_i 指向 i-1
A = causal_attn_weights(q_t, k_t)
assert A.shape == (n, n)
assert np.allclose(A.sum(-1), 1.0)                                    # 每行是概率分布
assert np.allclose(np.triu(A, k=1), 0.0, atol=1e-6)                   # 未来位置严格为 0
assert np.allclose(A[np.arange(1, n), np.arange(n - 1)], 1.0, atol=1e-2)  # 近似 one-hot 打到 i-1
assert np.allclose(A[0], np.eye(n)[0])                                # 第 0 行只能看自己
xn = rms_norm(W_E[tokens] + W_P[np.arange(len(tokens))])
assert np.allclose(causal_attn_weights(xn @ W_Q0, xn @ W_K0), cache["attn"][0], atol=1e-8)  # 复现 layer0
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `logit_lens`

实现 `logit_lens(resid_stack, W_U, gamma=None)`：输入 `(n_stages, n, d_model)` 的 residual 堆栈，
对每个阶段做 final RMSNorm（用给定 `gamma`）再乘 `W_U`，返回 `(n_stages, n, vocab)` 的逐层 logits。

**提示**：`rms_norm` 在最后一维归一化且支持任意前导维度广播——整个 stack 一行算完，函数体 1–2 行即可。
正确性的金标准自检：**末阶段的 lens 输出必须与模型真实 logits 完全一致**（lens 实现的标准单元测试）。

In [ ]:
def logit_lens(resid_stack, W_U, gamma=None):
    # TODO: 对 resid_stack 整体做 rms_norm(·, gamma) 再 @ W_U，返回 (n_stages, n, vocab)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
lens_logits = logit_lens(resid_stack, W_U)
assert lens_logits.shape == (3, len(tokens), V)
assert np.allclose(lens_logits[-1], logits, atol=1e-8)            # 末阶段 = 模型真实输出
assert (lens_logits[0].argmax(-1) == tokens).all()                # embedding 阶段：lens 预测当前 token
assert (lens_logits[1].argmax(-1) == tokens).all()                # +layer0：lens 看不到任何变化（正交写入）
assert (lens_logits[2, 7:13].argmax(-1) == tokens[8:14]).all()    # +layer1：induction 预测成形
assert np.allclose(logit_lens(resid_stack, W_U, np.ones(D)), lens_logits)  # gamma=1 与 None 等价
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `direct_logit_attribution`

实现 `direct_logit_attribution(components, final_resid, W_U)`：输入 `(n_comp, n, d_model)` 的组件写入堆栈
（这里是 `[embedding, head0, head1]`）与最终 residual `(n, d_model)`，返回 `(n_comp, n, vocab)`——
每个组件对每个 logit 的直接贡献，满足**各组件贡献之和 ≈ 最终 logits**。

**提示**：冻结尺度 `scale = 1 / sqrt(mean(final_resid**2, axis=-1, keepdims=True) + 1e-6)`（按位置一个标量），
然后 `(components * scale) @ W_U`。约 4 行。注意 `eps=1e-6` 必须与 `rms_norm` 一致，否则与 `logits` 的 `allclose` 过不了。

In [ ]:
def direct_logit_attribution(components, final_resid, W_U):
    # TODO: 用最终 residual 计算冻结的 1/rms（按位置, keepdims, eps=1e-6）
    # TODO: 返回 (components * scale) @ W_U，形状 (n_comp, n, vocab)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
comps = np.stack(cache["writes"])
dla = direct_logit_attribution(comps, cache["resid"][-1], W_U)
assert dla.shape == (3, len(tokens), V)
assert np.allclose(dla.sum(0), logits, atol=1e-6)            # 分解恒等式：贡献之和 = 最终 logits
assert np.abs(dla[1]).max() < 1e-9                           # head0 的直接贡献严格为 0（正交写入）
tgt = targets[7]
assert dla[2, 7, tgt] > 3.0                                  # head1 是正确答案 logit 的主要来源
assert dla[2, 7, tgt] > dla[0, 7, tgt] + 1.0                 # 且远大于 direct path 的贡献
print("✅ 练习 3 通过")

## ✏️ 练习 4：实现 `kl_to_final` —— 信念收敛曲线

量化"预测逐层成形"：实现 `kl_to_final(lens_logits, positions)`，对每个阶段 $\ell$ 计算
$\mathrm{KL}\big(p^{(\ell)} \,\|\, p^{(L)}\big)$ 在给定 positions 上的平均值，返回形状 `(n_stages,)`。
其中 $p^{(\ell)} = \mathrm{softmax}(\mathrm{lens}^{(\ell)})$，$p^{(L)}$ 是末阶段（= 模型真实分布）。

**提示**：`KL(p||q) = (p * (log p - log q)).sum(-1)`；先取 `lens_logits[:, positions, :]` 再 softmax；
对位置维取平均。约 5 行。softmax 输出严格 > 0，无需 clip。预期：末阶段 KL **恰为 0**，且随层数单调下降
（[Belrose 2023] 用同样的度量画 tuned lens 的收敛曲线）。

In [ ]:
def kl_to_final(lens_logits, positions):
    # TODO: p = softmax(lens_logits[:, positions, :])，q = p[-1]
    # TODO: 返回 KL(p_l || q) 对 positions 的平均，形状 (n_stages,)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
lens_all = rms_norm(resid_stack) @ W_U
kl = kl_to_final(lens_all, np.arange(7, 14))      # 只看第二遍序列（induction 生效区）
assert kl.shape == (3,)
assert kl[-1] < 1e-9                              # 末阶段与自己比：KL = 0
assert (kl >= -1e-12).all()                       # KL 非负
assert all(kl[i + 1] <= kl[i] + 1e-9 for i in range(2))   # 随层数单调下降
assert kl[0] > 1.0                                # 浅层信念与最终分布相去甚远
print("各阶段 KL(lens || final):", np.round(kl, 4))
print("✅ 练习 4 通过")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def causal_attn_weights(q, k):
    n, dh = q.shape
    scores = q @ k.T / np.sqrt(dh)
    mask = np.triu(np.ones((n, n), dtype=bool), k=1)
    scores = np.where(mask, -1e9, scores)
    return softmax(scores, axis=-1)

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def logit_lens(resid_stack, W_U, gamma=None):
    return rms_norm(resid_stack, gamma) @ W_U

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def direct_logit_attribution(components, final_resid, W_U):
    scale = 1.0 / np.sqrt((final_resid ** 2).mean(axis=-1, keepdims=True) + 1e-6)
    return (components * scale) @ W_U

In [ ]:
# 练习 4 参考答案（先自己做，再对照）
def kl_to_final(lens_logits, positions):
    p = softmax(lens_logits[:, positions, :], axis=-1)
    q = p[-1]
    return (p * (np.log(p) - np.log(q))).sum(-1).mean(-1)

---
## 🎯 真实数据胶囊题：真实 GPT-2 unembedding 上的 logit lens 解码

logit lens 把一个隐藏向量经 unembedding 投到词表、看它“指向哪些 token”。GPT-2 的输入 embedding 与 unembedding 绑定。用真实 wte 当 unembedding，实现把向量解码为 top-k token，验证一个 token 的 embedding 解码回它自己。

> 本模块新增的**真实数据**练习：用**真实 GPT-2 权重/embedding**把本章的可解释性技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, struct, urllib.request
import numpy as np
CACHE=os.path.expanduser("~/.interp_data"); os.makedirs(CACHE,exist_ok=True)
ST="https://huggingface.co/openai-community/gpt2/resolve/main/model.safetensors"
def _rng(s,e):
    req=urllib.request.Request(ST, headers={"Range":f"bytes={s}-{e}"})
    return urllib.request.urlopen(req,timeout=60).read()
def gpt2_emb_block(n=6000):
    cache=os.path.join(CACHE,f"wte_{n}.npy")
    if os.path.exists(cache): return np.load(cache)
    hlen=struct.unpack("<Q", _rng(0,7))[0]; hdr=json.loads(_rng(8,8+hlen-1))
    info=hdr["wte.weight"]; base=8+hlen; s0=info["data_offsets"][0]; d=info["shape"][1]
    raw=_rng(base+s0, base+s0+n*d*4-1)
    E=np.frombuffer(raw,dtype=np.float32).reshape(n,d).copy()
    np.save(cache,E); return E
def gpt2_vocab():
    p=os.path.join(CACHE,"vocab.json")
    if not os.path.exists(p): urllib.request.urlretrieve("https://huggingface.co/openai-community/gpt2/resolve/main/vocab.json",p)
    return json.load(open(p))
def digit_letter_dataset(lim=6000):
    "返回 (X[token嵌入], y[1=数字 0=字母], E, ids_digit, ids_alpha)"
    v=gpt2_vocab(); E=gpt2_emb_block(lim)
    dig=[i for t,i in v.items() if i<lim and t.isdigit()]
    alpha=[i for t,i in v.items() if i<lim and t.isalpha() and t.isascii()]
    rng=np.random.default_rng(0); alpha=list(rng.permutation(alpha)[:len(dig)])
    ids=dig+alpha; y=np.array([1]*len(dig)+[0]*len(alpha))
    return E[ids], y, E, dig, alpha
def shakespeare():
    p=os.path.join(CACHE,"shake.txt")
    if not os.path.exists(p): urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",p)
    return open(p).read()

E=gpt2_emb_block(6000); v=gpt2_vocab()
id2tok={i:t for t,i in v.items() if i<6000}
print("真实 unembedding E:", E.shape)

**练习**：实现 `logit_lens_topk(vec, E, k)`：返回与 `vec` 点积最大的前 k 个 token id。验证：用 token t 自己的 embedding 解码，t 应排第一（向量最像自己）。

In [ ]:
def logit_lens_topk(vec, E, k=5):
    # TODO: logits = E @ vec; 返回 logits 最大的 k 个索引(降序)
    raise NotImplementedError


In [ ]:
# 自测
for t in [15, 100, 500]:
    top=logit_lens_topk(E[t], E, 5)
    assert top[0]==t, f"token {t} 的 embedding 应解码回自己, 得到 {top[0]}"
print("logit lens ✓  token 自身 embedding 解码 top-1 = 自己")
print("示例 token 500 的近邻:", [id2tok.get(i,'?') for i in logit_lens_topk(E[500],E,5)])


### 📖 参考答案

In [ ]:
def logit_lens_topk(vec, E, k=5):
    logits = E @ vec
    return list(np.argsort(-logits)[:k])
print("✓ logit lens = 把中间表示投到词表读'它现在指向什么词'")